In [ ]:
#!/usr/bin/env python3
"""
════════════════════════════════════════════════════════════════════════════════
  RGCN + Drug-Conditioned Cross-Attention for Phenotype-Based Drug Repurposing
════════════════════════════════════════════════════════════════════════════════

Pipeline overview
─────────────────
  1. Load biomedical knowledge graph (nodes, edges, kg)
  2. Build train/test disease split; mask test-disease edges from the graph
  3. Construct typed edge index for R-GCN message passing
  4. (Optional) DistMult pre-training on the full KG link-prediction task
  5. Train RGCN + Cross-Attention scorer:
       • R-GCN encodes all KG nodes into dense embeddings
       • Drug embedding (query) attends over phenotype embeddings (keys/values)
       • Margin-ranking loss separates positive from negative drug–phenotype pairs
  6. Evaluate on held-out test diseases:
       Recall@1/5/10/50, MRR, AUROC, AUPRC (per-disease & balanced 1:1)
  7. Run ablation experiments:
       Exp 1 – PPR + Model ensemble at various interpolation weights
       Exp 2 – Residual R-GCN encoder
       Exp 3 – Lower LR (5e-4) + smaller margin (0.5)
       Exp 4 – Contraindication-aware hard-negative mining
       Exp 5 – Higher negative ratio (NEG_RATIO = 3)
       Exp 6 – InfoNCE contrastive loss
       Exp 7 – DistMult pre-train → finetune with 10 % val split
  8. PPR baseline for comparison
  9. Tail-drug evaluation (popularity-bias check)
  10. Error analysis & training curves

Data requirements
─────────────────
  Place the following files inside DATA_DIR:
    • nodes.csv          – columns: node_index, node_name, node_type
    • edges.csv          – edge list
    • kg.csv             – columns: x_index, y_index, relation
    • train_disease_ids.txt / test_disease_ids.txt
    • train_drug_pairs.csv / test_drug_pairs.csv   (disease_id, drug_id)

Hardware
────────
  A single GPU with ≥16 GB VRAM is recommended (tested on Colab T4 / A100).
  CPU-only works but will be very slow.
"""

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# §0  IMPORTS & REPRODUCIBILITY
# ══════════════════════════════════════════════════════════════════════════════
import gc
import json
import os
import random
import time
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")                       # non-interactive backend
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
import scipy.sparse as sp
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

# PyTorch Geometric – install with:
#   pip install torch-geometric torch-scatter torch-sparse
from torch_geometric.nn import RGCNConv

from sklearn.metrics import roc_auc_score, average_precision_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# §1  CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════
# ── Paths ────────────────────────────────────────────────────────────────────
DATA_DIR = "/content/drive/MyDrive/phenotype-drug/"

# ── Relation / node-type constants (match your kg.csv) ───────────────────────
INDICATION_REL = "indication"                   # drug → disease
PHENOTYPE_REL  = "disease_phenotype_positive"   # disease → phenotype
DRUG_TYPE      = "drug"
DISEASE_TYPE   = "disease"
PHENOTYPE_TYPE = "effect/phenotype"
SRC_COL        = "x_index"
DST_COL        = "y_index"
REL_COL        = "relation"

# ── Model hyper-parameters ───────────────────────────────────────────────────
HIDDEN_DIM   = 256          # node embedding dimension
NUM_LAYERS   = 3            # R-GCN layers
NUM_BASES    = 15           # basis decomposition rank for R-GCN
NUM_HEADS    = 4            # cross-attention heads
DROPOUT      = 0.2
LR           = 5e-4
WEIGHT_DECAY = 1e-5
MARGIN       = 0.5          # margin-ranking loss margin
NEG_RATIO    = 5            # negatives per positive
EPOCHS       = 200
PATIENCE     = 30           # early-stopping patience
EVAL_EVERY   = 10           # evaluate every N epochs
BATCH_SIZE   = 512
ACCUM_STEPS  = 4            # gradient-accumulation steps

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# §2  DATA LOADING
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("Loading data …")
print("=" * 60)

nodes = pd.read_csv(os.path.join(DATA_DIR, "nodes.csv"))
edges = pd.read_csv(os.path.join(DATA_DIR, "edges.csv"))
kg    = pd.read_csv(os.path.join(DATA_DIR, "kg.csv"))

print(f"Nodes: {nodes.shape}   Edges: {edges.shape}   KG triples: {kg.shape}")
print("Node types:\n", nodes["node_type"].value_counts().to_string())
print("Top relations:\n", kg[REL_COL].value_counts().head(15).to_string())

# ── Build disease→phenotype and disease→drug mappings ────────────────────────
phen_edges = kg[kg[REL_COL] == PHENOTYPE_REL]
disease_to_phenotypes = defaultdict(set)
for _, row in phen_edges.iterrows():
    disease_to_phenotypes[row[SRC_COL]].add(row[DST_COL])
    disease_to_phenotypes[row[DST_COL]].add(row[SRC_COL])   # treat as undirected

ind_edges = kg[kg[REL_COL] == INDICATION_REL]
disease_to_drugs = defaultdict(set)
for _, row in ind_edges.iterrows():
    disease_to_drugs[row[DST_COL]].add(row[SRC_COL])        # x=drug, y=disease

eligible = set(disease_to_phenotypes) & set(disease_to_drugs)
print(f"\nDiseases with ≥1 phenotype: {len(disease_to_phenotypes)}")
print(f"Diseases with ≥1 drug:      {len(disease_to_drugs)}")
print(f"Eligible (both):            {len(eligible)}")

# ── Train / test split ───────────────────────────────────────────────────────
train_diseases = set(
    int(x.strip())
    for x in open(os.path.join(DATA_DIR, "train_disease_ids.txt")).readlines()
)
test_diseases = set(
    int(x.strip())
    for x in open(os.path.join(DATA_DIR, "test_disease_ids.txt")).readlines()
)
train_pairs = pd.read_csv(os.path.join(DATA_DIR, "train_drug_pairs.csv"))
test_pairs  = pd.read_csv(os.path.join(DATA_DIR, "test_drug_pairs.csv"))

print(f"Train diseases: {len(train_diseases)}  |  Test diseases: {len(test_diseases)}")
print(f"Train pairs:    {len(train_pairs)}   |  Test pairs:    {len(test_pairs)}")

# Drug node index set
drug_indices     = set(nodes[nodes["node_type"] == DRUG_TYPE]["node_index"].tolist())
drug_indices_arr = np.array(sorted(drug_indices))
print(f"Total drug nodes: {len(drug_indices)}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# §3  GRAPH CONSTRUCTION  (mask out test-disease edges)
# ══════════════════════════════════════════════════════════════════════════════
# Remove any edge incident on a test disease to avoid leakage.
kg_train = kg[
    (~kg[SRC_COL].isin(test_diseases)) &
    (~kg[DST_COL].isin(test_diseases))
]
print(f"\nKG edges after masking test diseases: {len(kg_train):,}  (was {len(kg):,})")

# ── Relation encoding ────────────────────────────────────────────────────────
all_relations  = sorted(kg_train[REL_COL].unique().tolist())
rel2id         = {r: i for i, r in enumerate(all_relations)}
NUM_ORIG_RELS  = len(rel2id)

src_arr = kg_train[SRC_COL].values
dst_arr = kg_train[DST_COL].values
rel_arr = np.array([rel2id[r] for r in kg_train[REL_COL].values])

# Add reverse edges so R-GCN can pass messages in both directions.
edge_src = np.concatenate([src_arr, dst_arr])
edge_dst = np.concatenate([dst_arr, src_arr])
edge_rel = np.concatenate([rel_arr, rel_arr + NUM_ORIG_RELS])

NUM_RELATIONS = NUM_ORIG_RELS * 2
NUM_NODES     = len(nodes)

edge_index = torch.tensor(np.stack([edge_src, edge_dst]), dtype=torch.long, device=DEVICE)
edge_type  = torch.tensor(edge_rel, dtype=torch.long, device=DEVICE)

print(f"Nodes: {NUM_NODES:,}")
print(f"Edges (with reverse): {edge_index.shape[1]:,}")
print(f"Relation types (×2): {NUM_RELATIONS}")

# ── Row-normalised adjacency for PPR baseline ────────────────────────────────
rows_ppr = np.concatenate([src_arr, dst_arr])
cols_ppr = np.concatenate([dst_arr, src_arr])
data_ppr = np.ones(len(rows_ppr), dtype=np.float32)
A = csr_matrix((data_ppr, (rows_ppr, cols_ppr)), shape=(NUM_NODES, NUM_NODES))
row_sums = np.array(A.sum(axis=1)).flatten()
row_sums[row_sums == 0] = 1
A_norm = sp.diags(1.0 / row_sums) @ A
print(f"PPR adjacency built — nnz: {A_norm.nnz:,}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# §4  SUPERVISION LABELS
# ══════════════════════════════════════════════════════════════════════════════
train_disease_to_drugs = defaultdict(set)
for _, row in train_pairs.iterrows():
    train_disease_to_drugs[int(row["disease_id"])].add(int(row["drug_id"]))

test_disease_to_drugs = defaultdict(set)
for _, row in test_pairs.iterrows():
    test_disease_to_drugs[int(row["disease_id"])].add(int(row["drug_id"]))

# Flatten training samples: (disease_idx, phenotype_list, positive_drug)
train_samples = []
for d_idx in train_diseases:
    phenos = list(disease_to_phenotypes.get(d_idx, []))
    drugs  = list(train_disease_to_drugs.get(d_idx, []))
    if not phenos or not drugs:
        continue
    for drug in drugs:
        train_samples.append((d_idx, phenos, drug))

# ── Add off-label use edges as extra positives ───────────────────────────────
offlabel = kg_train[kg_train[REL_COL] == "off-label use"]
offlabel_count = 0
for _, row in offlabel.iterrows():
    d_idx = int(row[DST_COL])
    drug  = int(row[SRC_COL])
    phenos = list(disease_to_phenotypes.get(d_idx, []))
    if phenos and drug in drug_indices and d_idx not in test_diseases:
        train_samples.append((d_idx, phenos, drug))
        offlabel_count += 1

# Also enrich test ground truth with off-label
offlabel_all = kg[kg[REL_COL] == "off-label use"]
for _, row in offlabel_all.iterrows():
    d_idx = int(row[DST_COL])
    drug  = int(row[SRC_COL])
    if d_idx in test_diseases and drug in drug_indices:
        test_disease_to_drugs[d_idx].add(drug)

print(f"\nTraining samples (incl. {offlabel_count} off-label): {len(train_samples)}")

# ── Degree-biased negative-sampling weights ──────────────────────────────────
drug_degree = defaultdict(int)
for _, row in train_pairs.iterrows():
    drug_degree[int(row["drug_id"])] += 1

drug_list_sorted = sorted(drug_indices)
drug_weights = np.array([drug_degree.get(d, 0) + 1 for d in drug_list_sorted],
                        dtype=np.float64)
drug_weights /= drug_weights.sum()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# §5  METRICS
# ══════════════════════════════════════════════════════════════════════════════

def recall_at_k(ranked_drugs, true_drugs, k):
    """Fraction of true drugs found in the top-k of the ranked list."""
    top_k = set(ranked_drugs[:k])
    return len(top_k & set(true_drugs)) / len(true_drugs) if true_drugs else 0.0


def reciprocal_rank(ranked_drugs, true_drugs):
    """Reciprocal rank of the *first* true drug in the ranked list."""
    true_set = set(true_drugs)
    for rank, d in enumerate(ranked_drugs, start=1):
        if d in true_set:
            return 1.0 / rank
    return 0.0


def ppr_scores(seed_indices, A_norm, alpha=0.15, max_iter=50, tol=1e-6):
    """
    Personalised PageRank via power iteration.

    Parameters
    ----------
    seed_indices : list[int]   phenotype node indices (restart distribution)
    A_norm       : csr_matrix  row-normalised adjacency (N×N)
    alpha        : float       restart probability
    max_iter     : int         iteration cap
    tol          : float       L1 convergence threshold

    Returns
    -------
    r : ndarray (N,)  PPR score for every node
    """
    N = A_norm.shape[0]
    s = np.zeros(N, dtype=np.float32)
    if len(seed_indices) == 0:
        return s
    s[seed_indices] = 1.0 / len(seed_indices)
    r = s.copy()
    A_T = A_norm.T
    for _ in range(max_iter):
        r_new = (1 - alpha) * A_T.dot(r) + alpha * s
        if np.linalg.norm(r_new - r, 1) < tol:
            break
        r = r_new
    return r_new

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# §6  MODEL DEFINITION
# ══════════════════════════════════════════════════════════════════════════════

class DrugConditionedCrossAttention(nn.Module):
    """
    Drug embedding  = query   (batch, 1, dim)
    Phenotype embeds = keys/values  (batch, max_phenos, dim)
    → scalar relevance score per (drug, phenotype-set) pair
    """
    def __init__(self, dim, num_heads=4, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            dim, num_heads, dropout=dropout, batch_first=True
        )
        self.score_proj = nn.Linear(dim, 1)

    def forward(self, drug_emb, pheno_embs, pheno_mask=None):
        """
        drug_emb   : (B, dim)
        pheno_embs : (B, max_phenos, dim)
        pheno_mask : (B, max_phenos)  True = padded position
        Returns    : (B,) scores
        """
        query = drug_emb.unsqueeze(1)                             # (B, 1, dim)
        attn_out, _ = self.attn(
            query, pheno_embs, pheno_embs,
            key_padding_mask=pheno_mask
        )                                                         # (B, 1, dim)
        return self.score_proj(attn_out.squeeze(1)).squeeze(-1)   # (B,)


class PhenoDrugModel(nn.Module):
    """
    R-GCN encoder  →  drug-conditioned cross-attention scorer.

    The encoder runs message passing over the full heterogeneous KG
    (with basis decomposition for parameter sharing across relation types).
    The scorer uses the drug embedding as a query to attend over the
    phenotype embeddings associated with a query disease, producing a
    scalar compatibility score.
    """
    def __init__(self, num_nodes, num_relations, hidden_dim,
                 num_bases=10, num_layers=2, num_heads=4, dropout=0.2):
        super().__init__()
        # Learnable initial node features
        self.node_emb = nn.Embedding(num_nodes, hidden_dim)
        nn.init.xavier_uniform_(self.node_emb.weight)

        # Stacked R-GCN layers with LayerNorm + residual connections
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            self.convs.append(
                RGCNConv(hidden_dim, hidden_dim,
                         num_relations=num_relations, num_bases=num_bases)
            )
            self.norms.append(nn.LayerNorm(hidden_dim))

        self.cross_attn = DrugConditionedCrossAttention(hidden_dim, num_heads, dropout)
        self.dropout = dropout

    def encode(self, edge_index, edge_type):
        """Full-graph R-GCN forward → (num_nodes, hidden_dim)."""
        x = self.node_emb.weight
        for conv, norm in zip(self.convs, self.norms):
            residual = x
            x = conv(x, edge_index, edge_type)
            x = norm(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = x + residual                  # residual connection
        return x

    def score(self, node_embs, drug_idx, pheno_idx, pheno_mask):
        """
        node_embs : (N, dim)
        drug_idx  : (B,) LongTensor
        pheno_idx : (B, max_phenos) LongTensor – padded
        pheno_mask: (B, max_phenos) BoolTensor – True where padded
        Returns   : (B,) scores
        """
        drug_emb   = node_embs[drug_idx]        # (B, dim)
        pheno_embs = node_embs[pheno_idx]        # (B, max_phenos, dim)
        return self.cross_attn(drug_emb, pheno_embs, pheno_mask)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# §7  TRAINING UTILITIES
# ══════════════════════════════════════════════════════════════════════════════

def pad_pheno_batch(pheno_lists, device):
    """Pad variable-length phenotype index lists → (B, max_len) + mask."""
    max_len = max(len(p) for p in pheno_lists)
    B = len(pheno_lists)
    padded = torch.zeros(B, max_len, dtype=torch.long, device=device)
    mask   = torch.ones(B, max_len, dtype=torch.bool, device=device)
    for i, p in enumerate(pheno_lists):
        padded[i, :len(p)] = torch.tensor(p, dtype=torch.long)
        mask[i, :len(p)]   = False
    return padded, mask


def sample_negatives(positive_drug, true_drugs_set, n=1):
    """Degree-weighted negative drug sampling."""
    negs = []
    while len(negs) < n:
        candidates = np.random.choice(drug_list_sorted, size=n * 2, p=drug_weights)
        for c in candidates:
            if c not in true_drugs_set and c != positive_drug:
                negs.append(c)
                if len(negs) == n:
                    break
    return negs


def make_batch(samples, batch_indices, neg_ratio=NEG_RATIO):
    """Assemble a mini-batch of (pos_drug, neg_drug, phenotype-set) triples."""
    pos_drugs, neg_drugs, pheno_lists = [], [], []
    for idx in batch_indices:
        d_idx, phenos, pos_drug = samples[idx]
        true_set = train_disease_to_drugs.get(d_idx, set())
        negs = sample_negatives(pos_drug, true_set, neg_ratio)
        for n in negs:
            pos_drugs.append(pos_drug)
            neg_drugs.append(n)
            pheno_lists.append(phenos)
    pheno_padded, pheno_mask = pad_pheno_batch(pheno_lists, DEVICE)
    pos_t = torch.tensor(pos_drugs, dtype=torch.long, device=DEVICE)
    neg_t = torch.tensor(neg_drugs, dtype=torch.long, device=DEVICE)
    return pos_t, neg_t, pheno_padded, pheno_mask


def evaluate_model(model, diseases, disease_to_drugs_map, desc="Eval"):
    """Rank all drugs for each disease and compute metrics."""
    model.eval()
    results = []
    with torch.no_grad():
        node_embs = model.encode(edge_index, edge_type)
        all_drugs_t = torch.tensor(drug_indices_arr, dtype=torch.long, device=DEVICE)
        for disease_idx in tqdm(diseases, desc=desc):
            phenos     = list(disease_to_phenotypes.get(disease_idx, []))
            true_drugs = list(disease_to_drugs_map.get(disease_idx, []))
            if not phenos or not true_drugs:
                continue
            # Score all drugs in chunks to manage memory
            all_scores = []
            for cs in range(0, len(drug_indices_arr), 512):
                ce = min(cs + 512, len(drug_indices_arr))
                cp, cm = pad_pheno_batch([phenos] * (ce - cs), DEVICE)
                sc = model.score(node_embs, all_drugs_t[cs:ce], cp, cm)
                all_scores.append(sc.cpu().numpy())
            all_scores = np.concatenate(all_scores)
            ranked = drug_indices_arr[np.argsort(-all_scores)].tolist()
            results.append({
                "disease":  disease_idx,
                "n_seeds":  len(phenos),
                "n_true":   len(true_drugs),
                "R@1":      recall_at_k(ranked, true_drugs, 1),
                "R@5":      recall_at_k(ranked, true_drugs, 5),
                "R@10":     recall_at_k(ranked, true_drugs, 10),
                "R@50":     recall_at_k(ranked, true_drugs, 50),
                "MRR":      reciprocal_rank(ranked, true_drugs),
            })
    return pd.DataFrame(results)


def print_metrics(df, title="Results"):
    print("\n" + "=" * 50)
    print(title)
    print("=" * 50)
    for m in ["R@1", "R@5", "R@10", "R@50", "MRR"]:
        if m in df.columns:
            print(f"  {m:<6}: {df[m].mean():.4f}  ±  {df[m].std():.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# §8  PPR BASELINE
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 60)
print("PPR BASELINE")
print("═" * 60)

ppr_results = []
for disease_idx in tqdm(test_diseases, desc="PPR"):
    seeds      = list(disease_to_phenotypes.get(disease_idx, []))
    true_drugs = list(test_disease_to_drugs.get(disease_idx, []))
    if not seeds or not true_drugs:
        continue
    scores     = ppr_scores(seeds, A_norm, alpha=0.15)
    drug_sc    = scores[drug_indices_arr]
    ranked     = drug_indices_arr[np.argsort(-drug_sc)].tolist()
    ppr_results.append({
        "disease": disease_idx,
        "n_seeds": len(seeds), "n_true": len(true_drugs),
        "R@1":  recall_at_k(ranked, true_drugs, 1),
        "R@5":  recall_at_k(ranked, true_drugs, 5),
        "R@10": recall_at_k(ranked, true_drugs, 10),
        "R@50": recall_at_k(ranked, true_drugs, 50),
        "MRR":  reciprocal_rank(ranked, true_drugs),
    })

ppr_df = pd.DataFrame(ppr_results)
print_metrics(ppr_df, "PPR Baseline — Test Set")

# ── PPR alpha sensitivity (small sample) ─────────────────────────────────────
sample_diseases = list(test_diseases)[:30]
alpha_rows = []
for alpha in [0.05, 0.10, 0.15, 0.20, 0.30, 0.50, 0.75]:
    r1s, mrrs = [], []
    for d in sample_diseases:
        seeds = list(disease_to_phenotypes.get(d, []))
        td    = list(test_disease_to_drugs.get(d, []))
        if not seeds or not td:
            continue
        sc = ppr_scores(seeds, A_norm, alpha=alpha)
        rd = drug_indices_arr[np.argsort(-sc[drug_indices_arr])].tolist()
        r1s.append(recall_at_k(rd, td, 1))
        mrrs.append(reciprocal_rank(rd, td))
    alpha_rows.append({"alpha": alpha, "R@1": np.mean(r1s), "MRR": np.mean(mrrs)})
alpha_df = pd.DataFrame(alpha_rows)
print("\nPPR alpha sensitivity (30-disease sample):")
print(alpha_df.to_string(index=False))

# ── PPR balanced AUROC / AUPRC ───────────────────────────────────────────────
ppr_bal_labels, ppr_bal_scores = [], []
np.random.seed(42)
for d in tqdm(test_diseases, desc="PPR balanced"):
    ph = list(disease_to_phenotypes.get(d, []))
    td = set(test_disease_to_drugs.get(d, []))
    if not ph or not td:
        continue
    sc = ppr_scores(ph, A_norm, alpha=0.15)
    pos = list(td)
    neg = list(np.random.choice(
        [x for x in drug_indices_arr if x not in td],
        size=len(pos), replace=False
    ))
    ppr_bal_labels.extend([1] * len(pos) + [0] * len(neg))
    ppr_bal_scores.extend(sc[pos + neg].tolist())

print(f"\nPPR balanced AUPRC: {average_precision_score(ppr_bal_labels, ppr_bal_scores):.4f}")
print(f"PPR balanced AUROC: {roc_auc_score(ppr_bal_labels, ppr_bal_scores):.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# §9  MAIN TRAINING LOOP  (RGCN + CrossAttn)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 60)
print("TRAINING: R-GCN + Cross-Attention")
print("═" * 60)

model = PhenoDrugModel(
    num_nodes=NUM_NODES, num_relations=NUM_RELATIONS, hidden_dim=HIDDEN_DIM,
    num_bases=NUM_BASES, num_layers=NUM_LAYERS, num_heads=NUM_HEADS, dropout=DROPOUT,
).to(DEVICE)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=5
)
loss_fn = nn.MarginRankingLoss(margin=MARGIN)

best_mrr = 0.0
patience_counter = 0
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()
    perm = np.random.permutation(len(train_samples))
    epoch_loss, n_batches = 0.0, 0

    for start in range(0, len(perm), BATCH_SIZE):
        batch_idx = perm[start : start + BATCH_SIZE]
        pos_drugs, neg_drugs, pheno_padded, pheno_mask = make_batch(
            train_samples, batch_idx
        )
        node_embs  = model.encode(edge_index, edge_type)
        pos_scores = model.score(node_embs, pos_drugs, pheno_padded, pheno_mask)
        neg_scores = model.score(node_embs, neg_drugs, pheno_padded, pheno_mask)

        target = torch.ones_like(pos_scores)
        loss = loss_fn(pos_scores, neg_scores, target) / ACCUM_STEPS
        loss.backward()

        if (n_batches + 1) % ACCUM_STEPS == 0 or (start + BATCH_SIZE) >= len(perm):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()

        epoch_loss += loss.item() * ACCUM_STEPS
        n_batches  += 1

    avg_loss = epoch_loss / max(n_batches, 1)

    # ── Periodic evaluation ──────────────────────────────────────────────────
    if epoch % EVAL_EVERY == 0 or epoch == 1:
        eval_df = evaluate_model(model, test_diseases, test_disease_to_drugs,
                                 desc=f"Epoch {epoch}")
        mrr = eval_df["MRR"].mean()
        scheduler.step(mrr)
        history.append({
            "epoch": epoch, "loss": avg_loss,
            "R@1": eval_df["R@1"].mean(), "R@5": eval_df["R@5"].mean(),
            "R@10": eval_df["R@10"].mean(), "MRR": mrr,
        })
        print(f"  Epoch {epoch:3d}  loss={avg_loss:.4f}  MRR={mrr:.4f}  "
              f"R@10={eval_df['R@10'].mean():.4f}")
        if mrr > best_mrr:
            best_mrr = mrr
            patience_counter = 0
            torch.save(model.state_dict(), os.path.join(DATA_DIR, "best_model.pt"))
        else:
            patience_counter += EVAL_EVERY
            if patience_counter >= PATIENCE:
                print(f"  Early stopping at epoch {epoch}")
                break

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# §10  FINAL EVALUATION WITH BEST MODEL
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 60)
print("FINAL EVALUATION")
print("═" * 60)

model.load_state_dict(torch.load(os.path.join(DATA_DIR, "best_model.pt")))
final_df = evaluate_model(model, test_diseases, test_disease_to_drugs, desc="Final eval")
print_metrics(final_df, "R-GCN + CrossAttn — Test Set")

# ── Per-disease AUROC / AUPRC ────────────────────────────────────────────────
auc_rows = []
model.eval()
with torch.no_grad():
    ne = model.encode(edge_index, edge_type)
    all_drugs_t = torch.tensor(drug_indices_arr, dtype=torch.long, device=DEVICE)
    for d in tqdm(test_diseases, desc="AUROC/AUPRC"):
        ph = list(disease_to_phenotypes.get(d, []))
        td = set(test_disease_to_drugs.get(d, []))
        if not ph or not td:
            continue
        sc = []
        for cs in range(0, len(drug_indices_arr), 512):
            ce = min(cs + 512, len(drug_indices_arr))
            cp, cm = pad_pheno_batch([ph] * (ce - cs), DEVICE)
            sc.append(model.score(ne, all_drugs_t[cs:ce], cp, cm).cpu().numpy())
        sc = np.concatenate(sc)
        labels = np.array([1 if x in td else 0 for x in drug_indices_arr])
        if 0 < labels.sum() < len(labels):
            auc_rows.append({
                "disease": d,
                "AUROC": roc_auc_score(labels, sc),
                "AUPRC": average_precision_score(labels, sc),
            })
auc_df = pd.DataFrame(auc_rows)
print(f"\nMean per-disease AUROC: {auc_df['AUROC'].mean():.4f}")
print(f"Mean per-disease AUPRC: {auc_df['AUPRC'].mean():.4f}")

# ── Balanced 1:1 AUPRC (TxGNN-style) ────────────────────────────────────────
bal_labels, bal_scores = [], []
np.random.seed(42)
with torch.no_grad():
    for d in tqdm(test_diseases, desc="Balanced AUPRC"):
        ph = list(disease_to_phenotypes.get(d, []))
        td = set(test_disease_to_drugs.get(d, []))
        if not ph or not td:
            continue
        pos = list(td)
        neg = list(np.random.choice(
            [x for x in drug_indices_arr if x not in td],
            size=len(pos), replace=False
        ))
        eval_drugs = pos + neg
        eval_t = torch.tensor(eval_drugs, dtype=torch.long, device=DEVICE)
        cp, cm = pad_pheno_batch([ph] * len(eval_drugs), DEVICE)
        sc = model.score(ne, eval_t, cp, cm).cpu().numpy()
        bal_labels.extend([1] * len(pos) + [0] * len(neg))
        bal_scores.extend(sc.tolist())

print(f"\nBalanced AUPRC (1:1): {average_precision_score(bal_labels, bal_scores):.4f}")
print(f"Balanced AUROC (1:1): {roc_auc_score(bal_labels, bal_scores):.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# §11  TAIL-DRUG EVALUATION  (popularity-bias check)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 60)
print("TAIL-DRUG EVALUATION")
print("═" * 60)

drug_train_count = defaultdict(int)
for _, row in train_pairs.iterrows():
    drug_train_count[int(row["drug_id"])] += 1

TAIL_THRESHOLD = 2   # drugs with ≤2 training indications

tail_rows = []
with torch.no_grad():
    for _, row in final_df.iterrows():
        d_idx      = int(row["disease"])
        true_drugs = list(test_disease_to_drugs.get(d_idx, []))
        tail_drugs = [d for d in true_drugs if drug_train_count.get(d, 0) <= TAIL_THRESHOLD]
        if not tail_drugs:
            continue
        phenos = list(disease_to_phenotypes.get(d_idx, []))
        sc = []
        for cs in range(0, len(drug_indices_arr), 512):
            ce = min(cs + 512, len(drug_indices_arr))
            cp, cm = pad_pheno_batch([phenos] * (ce - cs), DEVICE)
            sc.append(model.score(ne, all_drugs_t[cs:ce], cp, cm).cpu().numpy())
        sc = np.concatenate(sc)
        ranked = drug_indices_arr[np.argsort(-sc)].tolist()
        tail_rows.append({
            "disease": d_idx, "n_tail": len(tail_drugs),
            "R@10": recall_at_k(ranked, tail_drugs, 10),
            "MRR":  reciprocal_rank(ranked, tail_drugs),
        })

tail_df = pd.DataFrame(tail_rows)
print(f"Diseases with tail drugs: {len(tail_df)}")
print(f"  R@10: {tail_df['R@10'].mean():.4f}")
print(f"  MRR:  {tail_df['MRR'].mean():.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# §12  EXPERIMENT 1 — Ensemble (PPR + Model)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 60)
print("EXP 1: PPR + Model Ensemble")
print("═" * 60)

ens_rows = []
with torch.no_grad():
    for disease_idx in tqdm(test_diseases, desc="Ensemble"):
        phenos     = list(disease_to_phenotypes.get(disease_idx, []))
        true_drugs = list(test_disease_to_drugs.get(disease_idx, []))
        if not phenos or not true_drugs:
            continue
        # Model scores
        ms = []
        for cs in range(0, len(drug_indices_arr), 512):
            ce = min(cs + 512, len(drug_indices_arr))
            cp, cm = pad_pheno_batch([phenos] * (ce - cs), DEVICE)
            ms.append(model.score(ne, all_drugs_t[cs:ce], cp, cm).cpu().numpy())
        ms = np.concatenate(ms)
        # PPR scores
        ps = ppr_scores(phenos, A_norm, alpha=0.15)[drug_indices_arr]
        # Min-max normalise both
        ms_n = (ms - ms.min()) / (ms.max() - ms.min() + 1e-8)
        ps_n = (ps - ps.min()) / (ps.max() - ps.min() + 1e-8)
        for w in [0.9, 0.8, 0.7, 0.6, 0.5]:
            combined = w * ms_n + (1 - w) * ps_n
            ranked   = drug_indices_arr[np.argsort(-combined)].tolist()
            ens_rows.append({
                "weight": w, "disease": disease_idx,
                "R@10": recall_at_k(ranked, true_drugs, 10),
                "R@50": recall_at_k(ranked, true_drugs, 50),
                "MRR":  reciprocal_rank(ranked, true_drugs),
            })

ens_df = pd.DataFrame(ens_rows)
print(ens_df.groupby("weight")[["MRR", "R@10", "R@50"]].mean().to_string())

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# §13  EXPERIMENT 2 — Residual R-GCN (explicit ablation)
# ══════════════════════════════════════════════════════════════════════════════
# Note: The main PhenoDrugModel already uses residual connections.
# This experiment removes them so we can measure their contribution.

print("\n" + "═" * 60)
print("EXP 2: No-Residual R-GCN (ablation)")
print("═" * 60)

class PhenoDrugModelNoResidual(nn.Module):
    """Identical to PhenoDrugModel but WITHOUT residual connections."""
    def __init__(self, num_nodes, num_relations, hidden_dim,
                 num_bases=10, num_layers=2, num_heads=4, dropout=0.2):
        super().__init__()
        self.node_emb = nn.Embedding(num_nodes, hidden_dim)
        nn.init.xavier_uniform_(self.node_emb.weight)
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            self.convs.append(RGCNConv(hidden_dim, hidden_dim,
                                       num_relations=num_relations, num_bases=num_bases))
            self.norms.append(nn.LayerNorm(hidden_dim))
        self.cross_attn = DrugConditionedCrossAttention(hidden_dim, num_heads, dropout)
        self.dropout = dropout

    def encode(self, edge_index, edge_type):
        x = self.node_emb.weight
        for conv, norm in zip(self.convs, self.norms):
            x = conv(x, edge_index, edge_type)   # NO residual
            x = norm(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return x

    def score(self, node_embs, drug_idx, pheno_idx, pheno_mask):
        return self.cross_attn(node_embs[drug_idx], node_embs[pheno_idx], pheno_mask)


def run_experiment(ModelClass, exp_name, lr=LR, margin=MARGIN, neg_ratio=NEG_RATIO,
                   epochs=EPOCHS, patience=PATIENCE, batch_fn=None):
    """Generic training + evaluation harness for ablation experiments."""
    gc.collect(); torch.cuda.empty_cache()
    torch.manual_seed(42); np.random.seed(42)

    m = ModelClass(
        num_nodes=NUM_NODES, num_relations=NUM_RELATIONS, hidden_dim=HIDDEN_DIM,
        num_bases=NUM_BASES, num_layers=NUM_LAYERS, num_heads=NUM_HEADS, dropout=DROPOUT,
    ).to(DEVICE)
    print(f"  {exp_name} params: {sum(p.numel() for p in m.parameters()):,}")

    opt = torch.optim.Adam(m.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=5)
    lf  = nn.MarginRankingLoss(margin=margin)

    best, pat_ctr = 0.0, 0
    _batch_fn = batch_fn or (lambda s, idx: make_batch(s, idx, neg_ratio))

    for ep in range(1, epochs + 1):
        m.train(); opt.zero_grad()
        perm = np.random.permutation(len(train_samples))
        eloss, nb = 0.0, 0
        for st in range(0, len(perm), BATCH_SIZE):
            bi = perm[st:st + BATCH_SIZE]
            pd_, nd_, pp, pm = _batch_fn(train_samples, bi)
            ne_ = m.encode(edge_index, edge_type)
            ps_ = m.score(ne_, pd_, pp, pm)
            ns_ = m.score(ne_, nd_, pp, pm)
            l = lf(ps_, ns_, torch.ones_like(ps_)) / ACCUM_STEPS
            l.backward()
            if (nb + 1) % ACCUM_STEPS == 0 or (st + BATCH_SIZE) >= len(perm):
                torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
                opt.step(); opt.zero_grad()
            eloss += l.item() * ACCUM_STEPS; nb += 1
        if ep % EVAL_EVERY == 0:
            edf = evaluate_model(m, test_diseases, test_disease_to_drugs, desc=f"{exp_name} ep{ep}")
            mrr = edf["MRR"].mean()
            sch.step(mrr)
            print(f"    ep {ep:3d}  loss={eloss/max(nb,1):.4f}  MRR={mrr:.4f}")
            if mrr > best:
                best = mrr; pat_ctr = 0
            else:
                pat_ctr += EVAL_EVERY
                if pat_ctr >= patience:
                    print(f"    Early stop at {ep}")
                    break

    edf = evaluate_model(m, test_diseases, test_disease_to_drugs, desc=f"{exp_name} final")
    print_metrics(edf, exp_name)
    del m; gc.collect(); torch.cuda.empty_cache()
    return edf


exp2_df = run_experiment(PhenoDrugModelNoResidual, "Exp2: No-Residual")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# §14  EXPERIMENT 3 — Lower LR + Smaller Margin
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 60)
print("EXP 3: LR=5e-4, Margin=0.5 (no residual baseline)")
print("═" * 60)

exp3_df = run_experiment(PhenoDrugModelNoResidual, "Exp3: LR=5e-4 M=0.5",
                         lr=5e-4, margin=0.5, epochs=150, patience=20)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# §15  EXPERIMENT 4 — Contraindication Hard Negatives
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 60)
print("EXP 4: Contraindication Hard-Negative Mining")
print("═" * 60)

# Build disease → contraindicated drug set
contra = kg_train[kg_train[REL_COL] == "contraindication"]
contra_map = defaultdict(set)
for _, row in contra.iterrows():
    contra_map[int(row[DST_COL])].add(int(row[SRC_COL]))
print(f"  Diseases with contraindications: {len(contra_map)}")


def sample_negatives_contra(positive_drug, disease_idx, true_set, n=1):
    """30 % chance of sampling a contraindicated drug as a hard negative."""
    negs = []
    contras = list(contra_map.get(disease_idx, set()) - true_set)
    if contras and np.random.random() < 0.3:
        chosen = list(np.random.choice(contras, size=min(n, len(contras)), replace=False))
        negs.extend(chosen)
    while len(negs) < n:
        c = np.random.choice(drug_list_sorted, p=drug_weights)
        if c not in true_set and c != positive_drug:
            negs.append(c)
    return negs[:n]


def make_batch_contra(samples, batch_indices):
    pos_drugs, neg_drugs, pheno_lists = [], [], []
    for idx in batch_indices:
        d_idx, phenos, pos_drug = samples[idx]
        true_set = train_disease_to_drugs.get(d_idx, set())
        negs = sample_negatives_contra(pos_drug, d_idx, true_set, NEG_RATIO)
        for n in negs:
            pos_drugs.append(pos_drug)
            neg_drugs.append(n)
            pheno_lists.append(phenos)
    pp, pm = pad_pheno_batch(pheno_lists, DEVICE)
    return (torch.tensor(pos_drugs, dtype=torch.long, device=DEVICE),
            torch.tensor(neg_drugs, dtype=torch.long, device=DEVICE), pp, pm)


exp4_df = run_experiment(PhenoDrugModel, "Exp4: Contra Hard-Neg",
                         batch_fn=make_batch_contra)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# §16  EXPERIMENT 5 — Higher Negative Ratio (NEG=3)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 60)
print("EXP 5: NEG_RATIO = 3")
print("═" * 60)

exp5_df = run_experiment(PhenoDrugModel, "Exp5: NEG=3", neg_ratio=3)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# §17  EXPERIMENT 6 — InfoNCE Loss
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 60)
print("EXP 6: InfoNCE Contrastive Loss")
print("═" * 60)

EXP6_NEG  = 5
EXP6_TEMP = 0.1


def make_batch_nce(samples, batch_indices):
    """Returns pos (B,), neg (B, num_neg), phenotype tensors."""
    all_pos, all_neg, pheno_lists = [], [], []
    for idx in batch_indices:
        d_idx, phenos, pos_drug = samples[idx]
        true_set = train_disease_to_drugs.get(d_idx, set())
        negs = sample_negatives(pos_drug, true_set, EXP6_NEG)
        all_pos.append(pos_drug)
        all_neg.append(negs)
        pheno_lists.append(phenos)
    pp, pm = pad_pheno_batch(pheno_lists, DEVICE)
    pos_t = torch.tensor(all_pos, dtype=torch.long, device=DEVICE)
    neg_t = torch.tensor(all_neg, dtype=torch.long, device=DEVICE)
    return pos_t, neg_t, pp, pm


def info_nce_loss(model, node_embs, pos_drugs, neg_drugs, pp, pm, temp=EXP6_TEMP):
    """InfoNCE: -log( exp(s+/τ) / [exp(s+/τ) + Σ exp(s-/τ)] )."""
    pos_sc = model.score(node_embs, pos_drugs, pp, pm)              # (B,)
    B, K = neg_drugs.shape
    neg_flat = neg_drugs.reshape(-1)
    pp_rep = pp.unsqueeze(1).expand(-1, K, -1).reshape(B * K, -1)
    pm_rep = pm.unsqueeze(1).expand(-1, K, -1).reshape(B * K, -1)
    neg_sc = model.score(node_embs, neg_flat, pp_rep, pm_rep).view(B, K)  # (B, K)
    logits = torch.cat([pos_sc.unsqueeze(1), neg_sc], dim=1) / temp       # (B, 1+K)
    labels = torch.zeros(B, dtype=torch.long, device=DEVICE)              # positive = index 0
    return F.cross_entropy(logits, labels)


# Train with InfoNCE
gc.collect(); torch.cuda.empty_cache()
torch.manual_seed(42); np.random.seed(42)

model_nce = PhenoDrugModel(
    num_nodes=NUM_NODES, num_relations=NUM_RELATIONS, hidden_dim=HIDDEN_DIM,
    num_bases=NUM_BASES, num_layers=NUM_LAYERS, num_heads=NUM_HEADS, dropout=DROPOUT,
).to(DEVICE)

opt_nce = torch.optim.Adam(model_nce.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sch_nce = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_nce, mode="max", factor=0.5, patience=5)

best_nce_mrr, pat_nce = 0.0, 0
for epoch in range(1, EPOCHS + 1):
    model_nce.train(); opt_nce.zero_grad()
    perm = np.random.permutation(len(train_samples))
    eloss, nb = 0.0, 0
    for st in range(0, len(perm), BATCH_SIZE):
        bi = perm[st:st + BATCH_SIZE]
        pos_t, neg_t, pp, pm = make_batch_nce(train_samples, bi)
        ne_ = model_nce.encode(edge_index, edge_type)
        loss = info_nce_loss(model_nce, ne_, pos_t, neg_t, pp, pm) / ACCUM_STEPS
        loss.backward()
        if (nb + 1) % ACCUM_STEPS == 0 or (st + BATCH_SIZE) >= len(perm):
            torch.nn.utils.clip_grad_norm_(model_nce.parameters(), 1.0)
            opt_nce.step(); opt_nce.zero_grad()
        eloss += loss.item() * ACCUM_STEPS; nb += 1
    if epoch % EVAL_EVERY == 0:
        edf = evaluate_model(model_nce, test_diseases, test_disease_to_drugs,
                             desc=f"NCE ep{epoch}")
        mrr = edf["MRR"].mean()
        sch_nce.step(mrr)
        print(f"  NCE ep {epoch:3d}  loss={eloss/max(nb,1):.4f}  MRR={mrr:.4f}")
        if mrr > best_nce_mrr:
            best_nce_mrr = mrr; pat_nce = 0
        else:
            pat_nce += EVAL_EVERY
            if pat_nce >= PATIENCE:
                print(f"  NCE early stop at {epoch}"); break

exp6_df = evaluate_model(model_nce, test_diseases, test_disease_to_drugs, desc="Exp6 final")
print_metrics(exp6_df, "Exp6: InfoNCE")
del model_nce; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# §18  EXPERIMENT 7 — DistMult Pre-train → Finetune (10 % val)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 60)
print("EXP 7: DistMult Pre-train → Finetune")
print("═" * 60)

# ── 7a. DistMult pre-training on the full KG (link prediction) ───────────────

class PretrainModelE2E(nn.Module):
    """R-GCN encoder + DistMult decoder for self-supervised KG pre-training."""
    def __init__(self, num_nodes, num_relations, hidden_dim,
                 num_bases=10, num_layers=2, dropout=0.2):
        super().__init__()
        self.node_emb = nn.Embedding(num_nodes, hidden_dim)
        nn.init.xavier_uniform_(self.node_emb.weight)
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            self.convs.append(RGCNConv(hidden_dim, hidden_dim,
                                       num_relations=num_relations, num_bases=num_bases))
            self.norms.append(nn.LayerNorm(hidden_dim))
        self.dropout = dropout
        self.rel_emb = nn.Embedding(num_relations, hidden_dim)
        nn.init.xavier_uniform_(self.rel_emb.weight)

    def encode(self, edge_index, edge_type):
        x = self.node_emb.weight
        for conv, norm in zip(self.convs, self.norms):
            x = conv(x, edge_index, edge_type)
            x = norm(x); x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return x

    def decode(self, node_embs, src, dst, rel):
        """DistMult score: s ⊙ r ⊙ d summed."""
        return (node_embs[src] * self.rel_emb(rel) * node_embs[dst]).sum(dim=-1)


gc.collect(); torch.cuda.empty_cache()
torch.manual_seed(42); np.random.seed(42)

pretrain_model = PretrainModelE2E(
    NUM_NODES, NUM_RELATIONS, HIDDEN_DIM, NUM_BASES, NUM_LAYERS, DROPOUT
).to(DEVICE)

# Pre-training edge arrays (forward + reverse)
pt_src = np.concatenate([kg_train[SRC_COL].values, kg_train[DST_COL].values])
pt_dst = np.concatenate([kg_train[DST_COL].values, kg_train[SRC_COL].values])
pt_rel_names = kg_train[REL_COL].values
pt_rel = np.concatenate([
    np.array([rel2id[r] for r in pt_rel_names]),
    np.array([rel2id[r] + NUM_ORIG_RELS for r in pt_rel_names]),
])
print(f"  Pre-train edges: {len(pt_src):,}")

PRETRAIN_EPOCHS = 10
DECODE_BS       = 8192
DECODES_PER_STEP = 4
pretrain_opt = torch.optim.Adam(pretrain_model.parameters(), lr=1e-3)
pretrain_lf  = nn.MarginRankingLoss(margin=1.0)

for ep in range(1, PRETRAIN_EPOCHS + 1):
    pretrain_model.train()
    perm = np.random.permutation(len(pt_src))
    eloss, ptr, n_steps = 0.0, 0, 0

    while ptr < len(perm):
        # Re-encode once per gradient step (amortise the cost)
        node_embs = pretrain_model.encode(edge_index, edge_type)
        pretrain_opt.zero_grad()
        step_loss = 0.0
        for _ in range(DECODES_PER_STEP):
            if ptr >= len(perm):
                break
            idx = perm[ptr : ptr + DECODE_BS]
            ptr += DECODE_BS
            s = torch.tensor(pt_src[idx], dtype=torch.long, device=DEVICE)
            d = torch.tensor(pt_dst[idx], dtype=torch.long, device=DEVICE)
            r = torch.tensor(pt_rel[idx], dtype=torch.long, device=DEVICE)
            nd = torch.randint(0, NUM_NODES, (len(idx),), device=DEVICE)
            pos = pretrain_model.decode(node_embs, s, d, r)
            neg = pretrain_model.decode(node_embs, s, nd, r)
            step_loss += pretrain_lf(pos, neg, torch.ones_like(pos))

        step_loss.backward()
        torch.nn.utils.clip_grad_norm_(pretrain_model.parameters(), 1.0)
        pretrain_opt.step()
        eloss += step_loss.item(); n_steps += 1

    print(f"  Pretrain ep {ep:2d}  loss={eloss/max(n_steps,1):.4f}")

# Save pre-trained encoder weights
pt_path = os.path.join(DATA_DIR, "pretrained_encoder.pt")
torch.save(pretrain_model.state_dict(), pt_path)
print(f"  Saved pre-trained weights → {pt_path}")
del pretrain_model; gc.collect(); torch.cuda.empty_cache()

# ── 7b. Finetune with 10 % validation split ─────────────────────────────────
train_disease_list = sorted(train_diseases)
np.random.seed(42)
np.random.shuffle(train_disease_list)
val_split       = int(0.1 * len(train_disease_list))
val_diseases_ft = set(train_disease_list[:val_split])
ft_diseases     = set(train_disease_list[val_split:])

# Val ground truth (indication + off-label)
val_disease_to_drugs_ft = defaultdict(set)
for _, row in train_pairs.iterrows():
    d = int(row["disease_id"])
    if d in val_diseases_ft:
        val_disease_to_drugs_ft[d].add(int(row["drug_id"]))
for _, row in offlabel_all.iterrows():
    d = int(row[DST_COL]); drug = int(row[SRC_COL])
    if d in val_diseases_ft and drug in drug_indices:
        val_disease_to_drugs_ft[d].add(drug)

# Finetune training samples
ft_samples = []
for d_idx in ft_diseases:
    phenos = list(disease_to_phenotypes.get(d_idx, []))
    drugs  = list(train_disease_to_drugs.get(d_idx, []))
    if not phenos or not drugs:
        continue
    for drug in drugs:
        ft_samples.append((d_idx, phenos, drug))
# Add off-label to finetune set
for _, row in offlabel.iterrows():
    d_idx = int(row[DST_COL]); drug = int(row[SRC_COL])
    phenos = list(disease_to_phenotypes.get(d_idx, []))
    if phenos and drug in drug_indices and d_idx in ft_diseases:
        ft_samples.append((d_idx, phenos, drug))

print(f"  FT train: {len(ft_diseases)} diseases, {len(ft_samples)} samples")
print(f"  Val:      {len(val_diseases_ft)} diseases")

# Build model, load pre-trained encoder (skip DistMult rel_emb)
model_ft = PhenoDrugModel(
    num_nodes=NUM_NODES, num_relations=NUM_RELATIONS, hidden_dim=HIDDEN_DIM,
    num_bases=NUM_BASES, num_layers=NUM_LAYERS, num_heads=NUM_HEADS, dropout=DROPOUT,
).to(DEVICE)

pretrained = torch.load(pt_path)
model_dict = model_ft.state_dict()
loaded = 0
for k, v in pretrained.items():
    if k in model_dict and v.shape == model_dict[k].shape and "rel_emb" not in k:
        model_dict[k] = v; loaded += 1
model_ft.load_state_dict(model_dict)
print(f"  Loaded {loaded} pre-trained params; cross-attention fresh")

opt_ft = torch.optim.Adam(model_ft.parameters(), lr=5e-4, weight_decay=1e-5)
sch_ft = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_ft, mode="max", factor=0.5, patience=5)
lf_ft  = nn.MarginRankingLoss(margin=MARGIN)
best_val, pat_ft = 0.0, 0

for epoch in range(1, EPOCHS + 1):
    model_ft.train(); opt_ft.zero_grad()
    perm = np.random.permutation(len(ft_samples))
    eloss, nb = 0.0, 0
    for st in range(0, len(perm), BATCH_SIZE):
        bi = perm[st:st + BATCH_SIZE]
        pd_, nd_, pp, pm = make_batch(ft_samples, bi)
        ne_ = model_ft.encode(edge_index, edge_type)
        ps_ = model_ft.score(ne_, pd_, pp, pm)
        ns_ = model_ft.score(ne_, nd_, pp, pm)
        l = lf_ft(ps_, ns_, torch.ones_like(ps_)) / ACCUM_STEPS
        l.backward()
        if (nb + 1) % ACCUM_STEPS == 0 or (st + BATCH_SIZE) >= len(perm):
            torch.nn.utils.clip_grad_norm_(model_ft.parameters(), 1.0)
            opt_ft.step(); opt_ft.zero_grad()
        eloss += l.item() * ACCUM_STEPS; nb += 1
    if epoch % EVAL_EVERY == 0:
        # Validate on held-out training diseases
        val_df = evaluate_model(model_ft, val_diseases_ft, val_disease_to_drugs_ft,
                                desc=f"FT val ep{epoch}")
        vmrr = val_df["MRR"].mean()
        sch_ft.step(vmrr)
        print(f"  FT ep {epoch:3d}  loss={eloss/max(nb,1):.4f}  val-MRR={vmrr:.4f}")
        if vmrr > best_val:
            best_val = vmrr; pat_ft = 0
            torch.save(model_ft.state_dict(), os.path.join(DATA_DIR, "best_ft_model.pt"))
        else:
            pat_ft += EVAL_EVERY
            if pat_ft >= PATIENCE:
                print(f"  FT early stop at {epoch}"); break

model_ft.load_state_dict(torch.load(os.path.join(DATA_DIR, "best_ft_model.pt")))
exp7_df = evaluate_model(model_ft, test_diseases, test_disease_to_drugs, desc="Exp7 final")
print_metrics(exp7_df, "Exp7: Pretrain + Finetune")
del model_ft; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# §19  TRAINING CURVES
# ══════════════════════════════════════════════════════════════════════════════
if history:
    hist_df = pd.DataFrame(history)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(hist_df["epoch"], hist_df["loss"], "o-")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.set_title("Training Loss")
    for m in ["R@1", "R@5", "R@10", "MRR"]:
        ax2.plot(hist_df["epoch"], hist_df[m], "o-", label=m)
    ppr_mrr = ppr_df["MRR"].mean()
    ax2.axhline(y=ppr_mrr, color="gray", ls="--", label=f"PPR MRR ({ppr_mrr:.4f})")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Score"); ax2.set_title("Test Metrics")
    ax2.legend()
    plt.tight_layout()
    fig.savefig(os.path.join(DATA_DIR, "training_curves.png"), dpi=150)
    print(f"\nTraining curves saved → {DATA_DIR}training_curves.png")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# §20  SAVE ALL RESULTS
# ══════════════════════════════════════════════════════════════════════════════
ppr_df.to_csv(os.path.join(DATA_DIR, "ppr_baseline_results.csv"), index=False)
alpha_df.to_csv(os.path.join(DATA_DIR, "ppr_alpha_sensitivity.csv"), index=False)
final_df.to_csv(os.path.join(DATA_DIR, "rgcn_crossattn_results.csv"), index=False)
tail_df.to_csv(os.path.join(DATA_DIR, "rgcn_tail_eval.csv"), index=False)
ens_df.to_csv(os.path.join(DATA_DIR, "ensemble_results.csv"), index=False)
if history:
    pd.DataFrame(history).to_csv(os.path.join(DATA_DIR, "training_history.csv"), index=False)
print("\nAll results saved. Done!")